In [1]:
!pip install -U langchain-ollama langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.2/467.2 kB 23.5 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.79
    Uninstalling langchain-core-0.3.79:
      Successfully uninstalled langchain-core-0.3.79
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.27 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.0.0 which is incompatible.


In [24]:
!pip uninstall -y langchain


Found existing installation: langchain 0.3.27
Uninstalling langchain-0.3.27:
  Successfully uninstalled langchain-0.3.27


In [25]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:

!ls /content/drive/MyDrive/review_dataset

sample_submission.csv  w_review_train.csv
test_file.csv	       w_review_train.parquet


In [27]:
import duckdb

con = duckdb.connect(database=':memory:', read_only=False)

In [28]:
df = con.execute("SELECT * FROM read_csv_auto('/content/drive/MyDrive/review_dataset/w_review_train.csv', parallel=false, encoding='UTF-8', ignore_errors=true)").fetchdf()

df.head()

,column0,column1
0,ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้...,3
1,อาหารที่นี่เป็นอาหารจีนแคะที่หากินยากในบ้านเรา...,4
2,ปอเปี๊ยะสด ทุกวันนี้รู้สึกว่าหากินยาก (ร้านที่...,3
3,รัานคัพเค้กในเมืองไทยมีไม่มาก หลายๆคนอาจจะสงสั...,5
4,อร่อย!!! เดินผ่านDigital gatewayทุกวัน ไม่ยักร...,5


In [29]:
df.shape

(40000, 2)

In [30]:
con.execute("COPY df TO '/content/drive/MyDrive/review_dataset/w_review_train.parquet' (FORMAT PARQUET)")

In [31]:
parquet_avg_rating_direct = con.execute("SELECT AVG(column1) FROM '/content/drive/MyDrive/review_dataset/w_review_train.parquet' WHERE column0 LIKE '%อร่อย%'").fetchone()[0]
parquet_avg_rating_direct

3.8313027179006562

In [32]:
parquet_file_path = '/content/drive/MyDrive/review_dataset/w_review_train.parquet'

In [33]:
parquet_avg_rating_direct = con.execute(f"SELECT AVG(column1) FROM '{parquet_file_path}' WHERE column0 LIKE '%อร่อย%'").fetchone()[0]
parquet_avg_rating_direct

3.8313027179006562

In [34]:
coffee_keywords = [
    'กาแฟ', 'คาเฟ่', 'coffee', 'cafe', 'เอสเปรสโซ่', 'ลาเต้',
    'มอคค่า', 'คาปูชิโน่', 'เฟรนช์เพรส', 'cold brew', 'drip',
    'espresso', 'latte', 'americano', 'macchiato', 'brew',
    'specialty', 'slow bar', 'ร้านกาแฟ', 'ร้านคาเฟ่', 'กาแฟสด',
    'เมนูกาแฟ', 'กาแฟนม', 'กาแฟดำ', 'ร้านบาริสต้า', 'barista', 'coffee shop'
]

where_clause = " OR ".join([f"column0 LIKE '%{keyword}%'" for keyword in coffee_keywords])

In [35]:
%%time
coffee_reviews_df = con.execute(f"SELECT * FROM '{parquet_file_path}' WHERE {where_clause}").fetchdf()

CPU times: user 1.84 s, sys: 61 ms, total: 1.91 s
Wall time: 1.91 s


In [36]:
csv_file_path = '/content/drive/MyDrive/review_dataset/w_review_train.csv'

In [37]:
%%time
coffee_reviews_df_csv = con.execute(f"SELECT * FROM read_csv_auto('{csv_file_path}', parallel=false, encoding='UTF-8', ignore_errors=true) WHERE {where_clause}").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CPU times: user 2.09 s, sys: 74.6 ms, total: 2.16 s
Wall time: 2.19 s


In [38]:
coffee_reviews_df.head()

,column0,column1
0,ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้...,3
1,วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอา...,4
2,สารภาพว่าไม่เคยคิดจะไปต่อคิวซื้อมากินเองครับ บ...,3
3,ร้านอาหารญี่ปุ่นร้านนี้ ใจจริงไม่อยากแนะนำเลยค...,5
4,เดือนแรกที่เค้าต่อคิวกัน 2 - 3 ชั่วโมง เราก็ว่...,5


In [39]:
coffee_reviews_df.shape

(4981, 2)

In [40]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [41]:
!nohup ollama serve &


nohup: appending output to 'nohup.out'


In [42]:
!ollama pull gemma3:270m

In [43]:
!ollama list

NAME           ID              SIZE      MODIFIED     
gemma3:270m    e7d36fb2c3b3    291 MB    1 second ago    


In [44]:
!ollama run gemma3:270m "สวัสดี"

สวัสดีค่ะ ยินดีที่ได้รู้จักค่ะ 😊




In [45]:
!curl -s http://127.0.0.1:11434/api/tags

{"models":[{"name":"gemma3:270m","model":"gemma3:270m","modified_at":"2025-10-18T04:37:30.569727116Z","size":291554930,"digest":"e7d36fb2c3b3293cfe56d55889867a064b3a2b22e98335f2e6e8a387e081d6be","details":{"parent_model":"","format":"gguf","family":"gemma3","families":["gemma3"],"parameter_size":"268.10M","quantization_level":"Q8_0"}}]}

In [47]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [48]:
llm = OllamaLLM(model="gemma3:270m")

In [49]:
response = llm.invoke("สวัสดี")
print(response)

สวัสดีครับ ยินดีที่ได้รู้จัก


In [50]:
template = """
คุณเป็นครูสอน Python
คำถาม : {question}
"""

In [51]:
prompt = PromptTemplate(
    input_variables=["question"],
    template=template
)

In [52]:
formatted_prompt = prompt.format(question="เขียนโค้ดหาผลรวมของตัวเลข 1 ถึง 10")
print(formatted_prompt)


คุณเป็นครูสอน Python
คำถาม : เขียนโค้ดหาผลรวมของตัวเลข 1 ถึง 10



In [53]:
chain = prompt | llm

In [54]:
response = chain.invoke({"question": "เขียนโค้ดหาผลรวมของตัวเลข 1 ถึง 10"})

In [55]:
response

'แน่นอนครับ! ผมจะช่วยคุณเขียนโค้ด Python เพื่อหาผลรวมของตัวเลข 1 ถึง 10 โดยใช้ตัวแปรและฟังก์ชัน `sum()`\n\n**โค้ด Python:**\n\n```python\ndef find_sum(numbers):\n  """\n  หาผลรวมของตัวเลข 1 ถึง 10\n  """\n  if not numbers:\n    return 0  # ถ้าไม่มีจำนวนเต็ม ให้ return 0\n\n  total = 0\n  for number in numbers:\n    total += number\n\n  return total\n\n# ตัวอย่างการใช้งาน\nnumbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]\nsum_of_numbers = find_sum(numbers)\nprint("ผลรวมของตัวเลข 1 ถึง 10:", sum_of_numbers)\n```\n\n**คำอธิบายโค้ด:**\n\n1.  **`def find_sum(numbers):`**:  กำหนดชื่อ funzione `find_sum` ที่จะทำการค้นหาผลรวมของตัวเลข\n2.  **`if not numbers:`**:  ตรวจสอบว่ามีตัวเลขใน `numbers`  เป็นจำนวนเต็มหรือไม่\n3.  **`total = 0`**:  ตั้งค่าค่า `total` เป็น 0\n4.  **`for number in numbers:`**:  วนซ้ำผ่านตัวเลขใน `numbers`\n5.  **`total += number`**:  เพิ่มค่า `number` เข้าไปใน `total`  (โดยใช้ตัวแวนต์ `+=`)\n6.  **`return total`**:  คืนค่าค่า `total`  ที่ได้จากการวนซ้ำ\n7.  **`numbers = [1, 2, 3,

In [56]:
from IPython.display import display, Markdown

In [57]:
display(Markdown(response))

แน่นอนครับ! ผมจะช่วยคุณเขียนโค้ด Python เพื่อหาผลรวมของตัวเลข 1 ถึง 10 โดยใช้ตัวแปรและฟังก์ชัน `sum()`

**โค้ด Python:**

```python
def find_sum(numbers):
  """
  หาผลรวมของตัวเลข 1 ถึง 10
  """
  if not numbers:
    return 0  # ถ้าไม่มีจำนวนเต็ม ให้ return 0

  total = 0
  for number in numbers:
    total += number

  return total

# ตัวอย่างการใช้งาน
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
sum_of_numbers = find_sum(numbers)
print("ผลรวมของตัวเลข 1 ถึง 10:", sum_of_numbers)
```

**คำอธิบายโค้ด:**

1.  **`def find_sum(numbers):`**:  กำหนดชื่อ funzione `find_sum` ที่จะทำการค้นหาผลรวมของตัวเลข
2.  **`if not numbers:`**:  ตรวจสอบว่ามีตัวเลขใน `numbers`  เป็นจำนวนเต็มหรือไม่
3.  **`total = 0`**:  ตั้งค่าค่า `total` เป็น 0
4.  **`for number in numbers:`**:  วนซ้ำผ่านตัวเลขใน `numbers`
5.  **`total += number`**:  เพิ่มค่า `number` เข้าไปใน `total`  (โดยใช้ตัวแวนต์ `+=`)
6.  **`return total`**:  คืนค่าค่า `total`  ที่ได้จากการวนซ้ำ
7.  **`numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]`**:  สร้างตัวแปร `numbers`  ที่มีค่า 1 ถึง 10
8.  **`sum_of_numbers = find_sum(numbers)`**:  เรียกใช้ฟังก์ชัน `find_sum`  เพื่อหาผลรวมของตัวเลขใน `numbers`
9.  **`print("ผลรวมของตัวเลข 1 ถึง 10:", sum_of_numbers)`**:  แสดงผลลัพธ์ของโค้ดให้ผู้ใช้เห็น

**ตัวอย่างการใช้งาน:**

```python
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
sum_of_numbers = find_sum(numbers)
print("ผลรวมของตัวเลข 1 ถึง 10:", sum_of_numbers)
```

**Output:**

```
ผลรวมของตัวเลข 1 ถึง 10: 55
```

**ข้อดีของโค้ดนี้:**

*   **เข้าใจง่าย:**  โค้ดนี้ใช้ตัวแวนต์ `+=` เพื่อเพิ่มค่าเข้าไปใน `total`  เมื่อจำเป็น
*   **Reusable:**  สามารถใช้ได้กับหลายๆ ตัวแปรและฟังก์ชัน
*   **Efficient:**  ใช้ตัวแวนต์ `+=`  เพื่อให้การเพิ่มค่าเข้าไปใน `total`  เป็นไปอย่างรวดเร็ว
*   **Clear:**  โค้ดนี้มีโครงสร้างที่ชัดเจน

หวังว่าโค้ดนี้จะเป็นประโยชน์! หากมีคำถามเพิ่มเติม ถามได้เลยครับ

In [58]:
prompt1 = PromptTemplate(
    template="แปลโจทย์เกี่ยวกับ {question}"
)

chain1 = prompt1 | llm

In [59]:
prompt2 = PromptTemplate(
    template="ทำความเข้าใจโจทย์ที่แปลแล้วต่อไปนี้ {translate}"
)

chain2 = prompt2 | llm

In [60]:
prompt3 = PromptTemplate(
    template="แก้โจทย์ตามที่เข้าใจต่อไปนี้ {understand}"
)

chain3 = prompt3 | llm

In [62]:
prompt4 = PromptTemplate(
    template="อธิบายวิธิีทำจากผลลัทธ์ต่อไปนี้ {solve}"
)

chain4 = prompt4 | llm

In [63]:
parser_chain = StrOutputParser()

In [64]:
full_chain = chain1 | chain2 | chain3 | chain4 | parser_chain

In [65]:
result = full_chain.invoke({"question":"2x + 5 = 15"})

In [66]:
result

'ผลลัทธ์เป็นแนวคิดที่สำคัญและมีบทบาทสำคัญในปรัคษะและวิชาการที่หลากหลาย การทำจากผลลัทธ์ถูกอธิบายอย่างละเอียด โดยพิจารณาจากหลักการพื้นฐานที่สำคัญดังนี้:\n\n* **การสร้างความเข้าใจ:** ผลลัทธ์ช่วยให้เราเข้าใจความสัมพันธ์ระหว่างปัจจัยต่างๆ (เช่น ความเชื่อ ความรู้สึก ความชอบ ความชอบธรรม) และผลลัพธ์ที่เกิดขึ้นจากความสัมพันธ์นั้นๆ\n* **การสร้างการตัดสินใจ:** ผลลัทธ์มีบทบาทในการกำหนดแนวทางและวิธีการในการตัดสินใจต่างๆ เช่น การเลือกใช้ทรัพยากร การวางแผนการทำงาน การตัดสินใจทางธุรกิจ\n* **การสร้างความสัมพันธ์:** ผลลัทธ์ช่วยให้เราสร้างความสัมพันธ์ที่ดีกับผู้อื่น ผ่านการสื่อสาร การแลกเปลี่ยนความคิดเห็น และการสร้างความรู้สึกเป็นส่วนหนึ่งของสังคม\n* **การพัฒนาตนเอง:** ผลลัทธ์เป็นเครื่องมือสำคัญในการพัฒนาตนเอง ผ่านการเรียนรู้ การคิดวิเคราะห์ และการสร้างสรรค์สิ่งใหม่ๆ\n* **การแก้ปัญหา:** ผลลัทธ์สามารถนำมาใช้ในการแก้ปัญหาต่างๆ โดยการวิเคราะห์ปัญหาและหาทางออกที่เหมาะสม\n\n**ความสำคัญของผลลัทธ์:**\n\nผลลัทธ์มีความสำคัญอย่างยิ่งในหลายบริบท ตั้งแต่การบริหารจัดการองค์กร ไปจนถึงการพัฒนาสังคม การทำจากผลลัทธ์มีประ

In [67]:
import os
from langchain_openai import ChatOpenAI

In [68]:
OPENAI_API_KEY = "xxx"

os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

In [69]:
llm2 = ChatOpenAI(model="gpt-4o-mini")

In [70]:
chain1 = prompt1 | llm2
chain2 = prompt2 | llm2
chain3 = prompt3 | llm2
chain4 = prompt4 | llm2

full_chain = chain1 | chain2 | chain3 | chain4 | parser_chain

In [71]:
formatted_result = result.replace('\\[', '$$') \
                         .replace('\\]', '$$') \
                         .replace('\\(', '$') \
                         .replace('\\)', '$')

In [72]:
display(Markdown(formatted_result))

ผลลัทธ์เป็นแนวคิดที่สำคัญและมีบทบาทสำคัญในปรัคษะและวิชาการที่หลากหลาย การทำจากผลลัทธ์ถูกอธิบายอย่างละเอียด โดยพิจารณาจากหลักการพื้นฐานที่สำคัญดังนี้:

* **การสร้างความเข้าใจ:** ผลลัทธ์ช่วยให้เราเข้าใจความสัมพันธ์ระหว่างปัจจัยต่างๆ (เช่น ความเชื่อ ความรู้สึก ความชอบ ความชอบธรรม) และผลลัพธ์ที่เกิดขึ้นจากความสัมพันธ์นั้นๆ
* **การสร้างการตัดสินใจ:** ผลลัทธ์มีบทบาทในการกำหนดแนวทางและวิธีการในการตัดสินใจต่างๆ เช่น การเลือกใช้ทรัพยากร การวางแผนการทำงาน การตัดสินใจทางธุรกิจ
* **การสร้างความสัมพันธ์:** ผลลัทธ์ช่วยให้เราสร้างความสัมพันธ์ที่ดีกับผู้อื่น ผ่านการสื่อสาร การแลกเปลี่ยนความคิดเห็น และการสร้างความรู้สึกเป็นส่วนหนึ่งของสังคม
* **การพัฒนาตนเอง:** ผลลัทธ์เป็นเครื่องมือสำคัญในการพัฒนาตนเอง ผ่านการเรียนรู้ การคิดวิเคราะห์ และการสร้างสรรค์สิ่งใหม่ๆ
* **การแก้ปัญหา:** ผลลัทธ์สามารถนำมาใช้ในการแก้ปัญหาต่างๆ โดยการวิเคราะห์ปัญหาและหาทางออกที่เหมาะสม

**ความสำคัญของผลลัทธ์:**

ผลลัทธ์มีความสำคัญอย่างยิ่งในหลายบริบท ตั้งแต่การบริหารจัดการองค์กร ไปจนถึงการพัฒนาสังคม การทำจากผลลัทธ์มีประโยชน์อย่างมากในการ:

* **การสร้างความเข้าใจ:** ช่วยให้เราเข้าใจความสัมพันธ์ระหว่างปัจจัยต่างๆ และสามารถตัดสินใจได้อย่างมีประสิทธิภาพ
* **การสร้างการตัดสินใจ:** ช่วยในการกำหนดแนวทางและวิธีการในการตัดสินใจต่างๆ
* **การสร้างความสัมพันธ์:** ช่วยในการสร้างความสัมพันธ์ที่ดีกับผู้อื่น
* **การพัฒนาตนเอง:** ช่วยในการพัฒนาตนเองผ่านการเรียนรู้ การคิดวิเคราะห์ และการสร้างสรรค์สิ่งใหม่ๆ
* **การแก้ปัญหา:** ช่วยในการวิเคราะห์ปัญหาและหาทางออกที่เหมาะสม
* **การพัฒนาสังคม:** ช่วยในการสร้างสังคมที่เอื้ออำนเรกความสุขและความเข้าใจ

**การทำจากผลลัทธ์ในแต่ละสาขา:**

* **การบริหารจัดการ:** การใช้ผลลัทธ์ในการวางแผน จัดการ และควบคุมทรัพยากรต่างๆ ในองค์กร
* **การพัฒนาธุรกิจ:** การใช้ผลลัทธ์ในการวิเคราะห์ปัญหา การหาทางออก และการตัดสินใจในธุรกิจ
* **การพัฒนาสังคม:** การใช้ผลลัทธ์ในการสร้างความสัมพันธ์ การสื่อสาร และการสร้างสังคม
* **การวิเคราะห์ปัญหา:** การใช้ผลลัทธ์ในการวิเคราะห์ปัญหาและหาทางออกที่เหมาะสม
* **การแก้ปัญหา:** การใช้ผลลัทธ์ในการวิเคราะห์ปัญหาและหาทางออกที่เหมาะสม

**สรุป:**

ผลลัทธ์เป็นเครื่องมือสำคัญที่ช่วยให้เราเข้าใจและสร้างความสัมพันธ์ที่ดีขึ้น การทำจากผลลัทธ์มีประโยชน์อย่างมากในหลากหลายสาขา รวมถึงการบริหารจัดการองค์กร การพัฒนาธุรกิจ และการพัฒนาสังคม

In [73]:
prompt_template = """
[1. Role/Context บอก LLM ว่าควรทำตัวเป็นอะไร]
คุณเป็น AI ที่เชี่ยวชาญในการวิเคราะห์รีวิวภาษาไทย

[2. Task Description บอกงานที่ต้องทำ]
จาก Review ต่อไปนี้ ให้ Extract 3 keywords
ที่เป็นตัวแทนของ Review นี้ได้ดีที่สุด

[3. Input Data ข้อมูลที่ต้องประมวลผล]
Review: {review}

[4. Output Format บอกรูปแบบของคำตอบ]
ตอบในรูปแบบ JSON โดยมีคีย์เป็น 'keywords'
เช่น {{"keywords": "กาแฟ, บรรยากาศ, ราคา"}

[5. Constraints ข้อจำกัดหรือเงื่อนไข]
- ต้องเป็นคำภาษาไทย
- คั่นด้วย comma
- หากไม่พบให้ตอบ 'ไม่ระบุ'

[6. Examples (optional) ตัวอย่าง input/output]
ตัวอย่าง
Input: "กาแฟอร่อย บรรยากาศดี ราคาแพง"
Output: {"keywords": "กาแฟ, บรรยากาศ, ราคา"}
"""

In [74]:
prompt = """
Extract keywords from this review
{review}
"""

# Pros Simple, fast
# Cons อาจได้ผลไม่ดีกับ Tasks ซับซ้อน

In [75]:
prompt = """
Extract keywords from reviews.

Example 1
Review: "กาแฟอร่อย บริการดี"
Keywords: กาแฟ, บริการ

Example 2
Review: "ราคาแพง แต่คุ้มค่า"
Keywords: ราคา, คุณภาพ

Now extract from this review
Review: {review}
Keywords:
"""

# Pros ได้ผลดีกว่า Zero-shot
# Cons Prompt ใช้ Tokens มากขึ้น

In [76]:
prompt = """
Review: {review}

ขั้นตอนการ Extract keywords
1. อ่านรีวิวและเข้าใจความหมาย
2. ระบุหัวข้อหลักที่กล่าวถึง
3. เลือก 3 keywords ที่สำคัญที่สุด
4. ตอบในรูปแบบ JSON: {{"keywords": "คำ1, คำ2, คำ3"}}

ให้ทำทีละขั้นตอน
"""

# Pros ได้ผลดีกับ Reasoning tasks
# Cons ใช้ Tokens มากกว่า ช้ากว่า

In [77]:
keyword_prompt = PromptTemplate(
template = """
จาก Review ต่อไปนี้
Extract 3 keywords ที่เป็นตัวแทนของ Review นี้

Review: {review}

ขั้นตอนการ Extract keywords
1. อ่านรีวิวและเข้าใจความหมาย
2. ระบุหัวข้อหลักที่กล่าวถึง
3. เลือก 3 keywords ที่สำคัญที่สุด
4. ตอบในรูปแบบ JSON โดยมี key คือ keyword เช่น :{{"keywords": "คำ1, คำ2, คำ3"}}

ให้ทำทีละขั้นตอน
""")

In [78]:
sample_reviews = coffee_reviews_df['column0'].head(2)
sample_reviews

,column0
0,ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้...
1,วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอา...


In [79]:
sample_reviews.to_csv('/content/sample_reviews.csv', index=False)

In [84]:
keyword_extract = keyword_prompt | llm | StrOutputParser()

In [89]:
%%time
results = []
for review in sample_reviews:
    text = keyword_extract.invoke({"review": review})
    cleaned = (text).replace("**", "").strip()
    results.append({"review": review, "text": cleaned})

CPU times: user 73.7 ms, sys: 2.26 ms, total: 76 ms
Wall time: 12.8 s


In [90]:
for result in results:
    print(f"Review {result['review'][:100]}...")
    print(f"text {result['text']}\n")

Review ร้านอาหารใหญ่มากกกกกกก 
เลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี 
ห้องทานหลักๆอยู่ชั้น 2 มีกาแฟ น้ำผึ...
text ```json
{
  "keywords": [
    "ร้านอาหารใหญ่",
    "ห้องน้ำ",
    "กาแฟ",
    "น้ำผึ้ง",
    "ไข่มัน",
    "คาร์บอนไฟ",
    "ขนมหวาน",
    "ฟรี",
    "ราคา",
    "กินฟรี",
    "ขนมไทย"
  ]
}
```

Review วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอารีย์

ก็ได้ยินมาบ่อยๆละ จากหลายที่ แต่ที่บ่อยสุดก็จาก ...
text ```json
{
  "keywords": [
    "ลาเต้",
    "ร้อน",
    "แก้วนึง",
    "ซดกาแฟ",
    "กาแฟ",
    "ร้านวาวี",
    "อารีย์",
    "Starbuck",
    "แฟนๆ Starbuck",
    "ตาข่าย",
    "กาแฟร้อน",
    "กาแฟดี",
    "กาแฟราคาไม่แพง",
    "กินบ่อยๆ",
    "จุกได้",
    "กาแฟกินได้",
    "กาแฟอร่อย",
    "กาแฟราคาดี",
    "กาแฟชื่นชอบ",
    "กาแฟสวย",
    "กาแฟมีค่า$$",
    "กาแฟดีใจ",
    "กาแฟอร่อย"
  ]
}
```



In [91]:
keyword_extract = keyword_prompt | llm2 | StrOutputParser()

In [92]:
for result in results:
    print(f"Review {result['review'][:100]}...")
    print(f"text {result['text']}\n")

Review ร้านอาหารใหญ่มากกกกกกก 
เลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี 
ห้องทานหลักๆอยู่ชั้น 2 มีกาแฟ น้ำผึ...
text ```json
{
  "keywords": [
    "ร้านอาหารใหญ่",
    "ห้องน้ำ",
    "กาแฟ",
    "น้ำผึ้ง",
    "ไข่มัน",
    "คาร์บอนไฟ",
    "ขนมหวาน",
    "ฟรี",
    "ราคา",
    "กินฟรี",
    "ขนมไทย"
  ]
}
```

Review วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอารีย์

ก็ได้ยินมาบ่อยๆละ จากหลายที่ แต่ที่บ่อยสุดก็จาก ...
text ```json
{
  "keywords": [
    "ลาเต้",
    "ร้อน",
    "แก้วนึง",
    "ซดกาแฟ",
    "กาแฟ",
    "ร้านวาวี",
    "อารีย์",
    "Starbuck",
    "แฟนๆ Starbuck",
    "ตาข่าย",
    "กาแฟร้อน",
    "กาแฟดี",
    "กาแฟราคาไม่แพง",
    "กินบ่อยๆ",
    "จุกได้",
    "กาแฟกินได้",
    "กาแฟอร่อย",
    "กาแฟราคาดี",
    "กาแฟชื่นชอบ",
    "กาแฟสวย",
    "กาแฟมีค่า$$",
    "กาแฟดีใจ",
    "กาแฟอร่อย"
  ]
}
```



In [93]:
!ollama pull scb10x/typhoon2.1-gemma3-4b

In [94]:
!ollama list

NAME                                  ID              SIZE      MODIFIED       
scb10x/typhoon2.1-gemma3-4b:latest    8cfab9097c9d    2.6 GB    14 minutes ago    
gemma3:270m                           e7d36fb2c3b3    291 MB    29 minutes ago    


In [95]:
llm3 = OllamaLLM(model="gemma3:270m")

In [96]:
keyword_extract = keyword_prompt | llm3 | StrOutputParser()

In [100]:
%%time
results = []
for review in sample_reviews:
    text = keyword_extract.invoke({"review": review})
    cleaned = (text).replace("**", "").strip()
    results.append({"review": review, "text": cleaned})

CPU times: user 50.9 ms, sys: 3.24 ms, total: 54.2 ms
Wall time: 5.85 s


In [101]:
for result in results:
    print(f"Review {result['review'][:100]}...")
    print(f"text {result['text']}\n")

Review ร้านอาหารใหญ่มากกกกกกก 
เลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี 
ห้องทานหลักๆอยู่ชั้น 2 มีกาแฟ น้ำผึ...
text ```json
"keywords": [
  "ร้านอาหารใหญ่",
  "ห้องน้ำ",
  "กาแฟ",
  "น้ำผึ้ง",
  "น้ำผึ้งมาราด",
  "แพงเวอร์",
  "ไม่สั่ง",
  "ลาบไข่ต้ม",
  "ไข่มันคาว",
  "ไม่ประทับใจ",
  "ราคา150บาท",
  "เกินไป"
]
```

Review วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอารีย์

ก็ได้ยินมาบ่อยๆละ จากหลายที่ แต่ที่บ่อยสุดก็จาก ...
text ```json
{
  "keywords": [
    "ลาเต้",
    "ร้อน",
    "แก้วนึง",
    "ร้อน",
    "มาวางอยู่ข้างหน้า",
    "ซากอารยธรรมลาเต้",
    "เมพพพพ",
    "รสชาติ",
    "กลิ่น",
    "มวยรุ่นเดียวกะ Starbuck",
    "อร่อย",
    "ราคาไม่แพง",
    "กินบ่อยก็จุกได้",
    "ซีเรียส",
    "ซอยไรจำไม่ได้",
    "แต่อยู่ตรงข้าม Villa",
    "55+",
    "ถ่ายโดย : Tan"
  ]
}
```



In [102]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser

class ReviewKeywords(BaseModel):
    keywords: str = Field(
        ...,  # Required field
        description="3 keywords separated by comma",
        examples=["กาแฟ, บรรยากาศ, ราคา"]
    )

class KeywordsExtraction(BaseModel):
    extracted_qualities: ReviewKeywords

In [103]:
keywords_parser = JsonOutputParser(pydantic_object=KeywordsExtraction)

In [104]:
keyword_extract = keyword_prompt | llm3 | keywords_parser

In [105]:
%%time
results = []
for review in sample_reviews:
    text = keyword_extract.invoke({"review": review})
    results.append({"review": review, "text": text})

CPU times: user 40 ms, sys: 4.03 ms, total: 44 ms
Wall time: 4.69 s


In [106]:
results[0]['text']['keywords']

['ร้านอาหารใหญ่',
 'ห้องน้ำ',
 'กาแฟ',
 'น้ำผึ้ง',
 'น้ำผึ้งมาราด',
 'แพง',
 'เกินไป',
 'รับไม่ไหว',
 'ว',
 'ฟรี',
 'ขนมไทย',
 'ขนมหวาน',
 'ค่อนข้าง',
 'ไม่ประทับใจ',
 'น่าเสียดาย',
 'ราคาสูง']

In [107]:
results[1]['text']['keywords']

['ลาเต้',
 'รสชาติ',
 'กลิ่น',
 'เมพพพ',
 'Chocolate',
 'ซอย',
 'หน้าตา',
 'ถ่ายโดย',
 'Tan']

In [108]:
llm3 = OllamaLLM(
    model="scb10x/typhoon2.1-gemma3-4b",
    temperature=0.05
)

keyword_extract = keyword_prompt | llm3 | keywords_parser

In [109]:
%%time
results = []
for review in sample_reviews:
    text = keyword_extract.invoke({"review": review})
    results.append({"review": review, "text": text})

CPU times: user 163 ms, sys: 13.2 ms, total: 176 ms
Wall time: 20.2 s


In [110]:
results[0]['text']['keywords'], results[1]['text']['keywords']

(['ราคา', 'รสชาติ', 'ขนาด'], ['กาแฟ', 'วาวี', 'อารีย์'])

In [111]:
keyword_prompt = PromptTemplate(
    template="""จาก Review ต่อไปนี้
    Extract keywords 3 keywords เท่านั้น ที่เป็นตัวแทนของ Review นี้ได้ดีที่สุด

    Review: {review}

    ตอบในรูปแบบ JSON โดยมีคีย์เป็น 'keywords'
    และค่าเป็นข้อความที่ดึงได้คั่นด้วย comma
    เช่น {{"keywords": "กาแฟ, บรรยากาศ, ราคา"}}

    หากไม่พบข้อมูลให้ตอบ 'ไม่ระบุ'
    """
)

In [112]:
keyword_extract = keyword_prompt | llm3 | keywords_parser

In [113]:
inputs = [{"review": r} for r in sample_reviews]

In [114]:
%%time
results = keyword_extract.batch(inputs)

CPU times: user 20.2 ms, sys: 2.39 ms, total: 22.6 ms
Wall time: 2.42 s


In [115]:
results

[{'keywords': 'ราคา, บรรยากาศ, ไข่ต้ม'},
 {'keywords': 'กาแฟ, บรรยากาศ, รสชาติ'}]

In [117]:
sample_reviews_1000 = coffee_reviews_df['column0'].head(100).tolist()
inputs_1000 = [{"review": r} for r in sample_reviews_1000]

BATCH_SIZE = 8

results_1000 = []

In [118]:
from tqdm import tqdm

In [119]:
for i in tqdm(range(0, len(inputs_1000), BATCH_SIZE), desc="Extracting keywords"):
    batch = inputs_1000[i:i+BATCH_SIZE]
    outputs = keyword_extract.batch(batch)
    for inp, out in zip(batch, outputs):
        results_1000.append({"review": inp["review"], "text": out})

Extracting keywords: 100%|██████████| 13/13 [02:10<00:00, 10.02s/it]


In [120]:
for result in results_1000[0:10]:
    print(result['text']['keywords'])

ราคา, บรรยากาศ, ไข่ต้ม
กาแฟ, บรรยากาศ, รสชาติ
โดนัท, รสชาติ, ความหวาน
วาซาบิ, ปลาซาบะ, ข้าวหน้าปลา
โดนัท, Original Glazed, คิว
วังพญาไท, นรสิงห์, คาเฟ่
อาหารทะเล, ราคา, บรรยากาศ
กาแฟ, เค้ก, พนักงาน
ฮันนี่ โทสท์, ราคา, รสชาติ
คาเฟ่, วานิลลา, คาโบนาร่า


In [121]:
  results_1000[0]

{'review': 'ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี \nห้องทานหลักๆอยู่ชั้น 2 มีกาแฟ น้ำผึ้ง ซึ่งก็แค่เอาน้ำผึ้งมาราด แพงเวอร์ อย่าสั่งเลย \nลาบไข่ต้ม ไข่มันคาวอะ เลยไม่ประทับใจเท่าไหร่\nทอดมันหัวปลีกรอบอร่อยต้องเบิ้ล \nพะแนงห่อไข่อร่อยดี เห้ยแต่ราคา 150บาทมันเกินไปนะ รับไม่ไหวว\nเลิกกินแล้วมีขนมหวานให้กินฟรีเล็กน้อย )ขนมไทย) \n\nคงไม่ไปซ้ำ แพงเกิน ',
 'text': {'keywords': 'ราคา, บรรยากาศ, ไข่ต้ม'}}

In [122]:
context_prompt = PromptTemplate(
    template="""จาก Review ต่อไปนี้
    Extract context ใน Review ที่เป็นตัวแทนที่บรรยายคำว่า {keyword} ได้ดีที่สุด

    Review: {review}

    ตอบผลลัพธ์ในรูปแบบ JSON โดยมีคีย์เป็น 'context'
    และค่าเป็นข้อความที่ดึงได้จาก Review

    ตัวอย่าง
    keyword คือ "กาแฟ"
    context: "กาแฟที่นี่อร่อยมาก หอมกลิ่นกาแฟคั่วสด"

    หากไม่พบข้อมูลให้ตอบ 'ไม่ระบุ'
    """
)

In [123]:
sentiment_prompt = PromptTemplate(
    template="""จากข้อความต่อไปนี้ ให้วิเคราะห์ว่าเป็น sentiment แบบไหน
    เลือกจากตัวเลือกเหล่านี้เท่านั้น: "positive", "negative", "neutral"

    ข้อความ: {message}

    ตอบในรูปแบบ JSON โดยมีคีย์เป็น 'sentiment'
    และค่าเป็นหนึ่งใน list ต่อไปนี้ [positive, negative, neutral]

    คำแนะนำ
    - positive คือ ข้อความที่แสดงความพอใจ ชอบ ดี
    - negative คือ ข้อความที่แสดงความไม่พอใจ ไม่ชอบ แย่
    - neutral คือ ข้อความที่ไม่แสดงความรู้สึกชัดเจน เป็นกลาง

    # ตัวอย่าง
    # ข้อความ "อร่อยมาก" → {{"sentiment": "positive"}}
    # ข้อความ "แพงไป" → {{"sentiment": "negative"}}
    # ข้อความ "ปกติ" → {{"sentiment": "neutral"}}
    """
)


In [124]:
class ContextKeyword(BaseModel):
    context: str = Field(..., description="context that best represent this keyword")

class ContexExtraction(BaseModel):
    extracted: ContextKeyword

context_parser = JsonOutputParser(pydantic_object=ContexExtraction)

In [125]:
context_extract = context_prompt | llm3 | context_parser

In [126]:
class Sentiment(BaseModel):
    sentiment: str = Field(..., description="sentiment of review")

class SentimentExtraction(BaseModel):
    extracted: Sentiment


sentiment_parser = JsonOutputParser(pydantic_object=SentimentExtraction)

In [127]:
sentiment_extract = sentiment_prompt | llm3 | sentiment_parser

In [128]:
keywords = results_1000[0]['text']['keywords']


In [129]:
context_inputs = []

for review_data in tqdm(results_1000, desc="Preparing context inputs"):
    review = review_data['review']
    keywords = review_data['text']['keywords']
    for keyword in keywords:
        context_inputs.append({"review": review, "keyword": keyword.strip()})

Preparing context inputs: 100%|██████████| 100/100 [00:00<00:00, 39960.98it/s]


In [130]:
len(context_inputs)

2524

In [133]:
context_results = []
for i in tqdm(range(len(results_1000)), desc="Extracting contexts"):
   result = context_extract.batch(context_inputs[i*3:(i*3)+3])
   contexts_list = [d['context'] for d in result]
   results_1000[i]['contexts'] = contexts_list

Extracting contexts: 100%|██████████| 100/100 [08:51<00:00,  5.32s/it]


In [134]:
results_1000[:2]

[{'review': 'ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี \nห้องทานหลักๆอยู่ชั้น 2 มีกาแฟ น้ำผึ้ง ซึ่งก็แค่เอาน้ำผึ้งมาราด แพงเวอร์ อย่าสั่งเลย \nลาบไข่ต้ม ไข่มันคาวอะ เลยไม่ประทับใจเท่าไหร่\nทอดมันหัวปลีกรอบอร่อยต้องเบิ้ล \nพะแนงห่อไข่อร่อยดี เห้ยแต่ราคา 150บาทมันเกินไปนะ รับไม่ไหวว\nเลิกกินแล้วมีขนมหวานให้กินฟรีเล็กน้อย )ขนมไทย) \n\nคงไม่ไปซ้ำ แพงเกิน ',
  'text': {'keywords': 'ราคา, บรรยากาศ, ไข่ต้ม'},
  'contexts': ['ร้านอาหารใหญ่มากกกกกกก',
   'ร้านอาหารใหญ่มากกกกกกก เลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี',
   'ห้องทานอาหารอยู่ชั้น 2 มีกาแฟ น้ำผึ้ง ซึ่งก็แค่เอาน้ำผึ้งมาราด แพงเวอร์ อย่าสั่งเลย']},
 {'review': 'วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอารีย์\n\nก็ได้ยินมาบ่อยๆละ จากหลายที่ แต่ที่บ่อยสุดก็จาก Twitter ว่ากาแฟที่นี่อร่อยมากกกกกกก\nเรียกว่าระดับแฟนๆ Starbuck ยังต้องเหลียวมามอง ก็ดองมานานจนถึงวันนี้โอกาสเหมาะ ไป\npresent งานที่ตึก Software Park เสร็จก็เลยมาหาไรกินแถวนี้พอดี\nจัดไป.....\n\n\nบรรยากาศรอบๆร้านก็แต่งได้น่านั่ง อารมณ์ประมาณว่าอยู่ในสว

In [135]:
sentiment_inputs = []

for review_data in tqdm(results_1000, desc="Preparing sentiment inputs"):
    contexts = review_data['contexts']
    for context in contexts:
        sentiment_inputs.append({"message": context})

Preparing sentiment inputs: 100%|██████████| 100/100 [00:00<00:00, 175567.35it/s]


In [136]:
len(sentiment_inputs)

300

In [137]:
sentiment_inputs[:6]

[{'message': 'ร้านอาหารใหญ่มากกกกกกก'},
 {'message': 'ร้านอาหารใหญ่มากกกกกกก เลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี'},
 {'message': 'ห้องทานอาหารอยู่ชั้น 2 มีกาแฟ น้ำผึ้ง ซึ่งก็แค่เอาน้ำผึ้งมาราด แพงเวอร์ อย่าสั่งเลย'},
 {'message': 'ร้านอาหารใหญ่มากกกกกกก เลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี'},
 {'message': 'ราคา 150 บาทมันเกินไปนะ รับไม่ไหวว'},
 {'message': 'ทอดมันหัวปลีกรอบอร่อยต้องเบิ้ล'}]

In [138]:
for i in tqdm(range(len(results_1000)), desc="Extracting sentiment"):
   result = sentiment_extract.batch(sentiment_inputs[i*3:(i*3)+3])
   sentiment_list = [d['sentiment'] for d in result]
   results_1000[i]['sentiment'] = sentiment_list

Extracting sentiment: 100%|██████████| 100/100 [04:21<00:00,  2.61s/it]


In [139]:
results_1000[0:2]

[{'review': 'ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี \nห้องทานหลักๆอยู่ชั้น 2 มีกาแฟ น้ำผึ้ง ซึ่งก็แค่เอาน้ำผึ้งมาราด แพงเวอร์ อย่าสั่งเลย \nลาบไข่ต้ม ไข่มันคาวอะ เลยไม่ประทับใจเท่าไหร่\nทอดมันหัวปลีกรอบอร่อยต้องเบิ้ล \nพะแนงห่อไข่อร่อยดี เห้ยแต่ราคา 150บาทมันเกินไปนะ รับไม่ไหวว\nเลิกกินแล้วมีขนมหวานให้กินฟรีเล็กน้อย )ขนมไทย) \n\nคงไม่ไปซ้ำ แพงเกิน ',
  'text': {'keywords': 'ราคา, บรรยากาศ, ไข่ต้ม'},
  'contexts': ['ร้านอาหารใหญ่มากกกกกกก',
   'ร้านอาหารใหญ่มากกกกกกก เลี้ยวเข้ามาเจอห้องน้ำก่อนเลย เออแปลกดี',
   'ห้องทานอาหารอยู่ชั้น 2 มีกาแฟ น้ำผึ้ง ซึ่งก็แค่เอาน้ำผึ้งมาราด แพงเวอร์ อย่าสั่งเลย'],
  'sentiment': ['neutral', 'negative', 'negative']},
 {'review': 'วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอารีย์\n\nก็ได้ยินมาบ่อยๆละ จากหลายที่ แต่ที่บ่อยสุดก็จาก Twitter ว่ากาแฟที่นี่อร่อยมากกกกกกก\nเรียกว่าระดับแฟนๆ Starbuck ยังต้องเหลียวมามอง ก็ดองมานานจนถึงวันนี้โอกาสเหมาะ ไป\npresent งานที่ตึก Software Park เสร็จก็เลยมาหาไรกินแถวนี้พอดี\nจัดไป.....\n\n\nบรรย

In [141]:
import pandas as pd

df = pd.DataFrame([
    {
        'review': r['review'],
        **{f'keyword{i+1}': k.strip() for i, k in enumerate(r['text']['keywords'])},
        **{f'context{i+1}_keyword': c for i, c in enumerate(r['contexts'])},
        **{f'sentiment{i+1}': s for i, s in enumerate(r['sentiment'])}
    }
    for r in results_1000
])

In [142]:
df.to_csv('results.csv', index=False, encoding='utf-8-sig')